# LLM relevance judge

For every (job, CV) pair of `1_application.ipynb` (`applications/applications_short.csv`), a local
Ollama LLM rates how well the candidate fits the job:

| score | meaning |
|---|---|
| 0 | not relevant: a different sector |
| 1 | partially relevant: right area, clear gaps |
| 2 | relevant: the occupation or field fits |

This is the ground-truth label of the pipeline; `1_application.ipynb` only marks the CVs of the
same category (`relevant_cvs`).

## Label

Each pair is judged K = 5 times at `temperature = 0.6`, so the runs can differ; the label is the
median of the 5 runs.

## Confidence

The judge is not asked for its confidence: it is measured from the agreement of the 5 runs, in
two columns computed from `run_1..run_5`:

| column | definition | values |
|---|---|---|
| `agreement` | share of runs equal to the label | 1.0 = 5/5, 0.8 = 4/5, 0.6 = 3/5 |
| `stability` | `1 - mean\|run - label\| / 2` | 1.0 = unanimous, 0.0 = every run two grades away |

`stability` separates runs with the same `agreement`: `(2,2,2,2,1)` and `(2,2,2,2,0)` both have
`agreement = 0.8`, and `stability` 0.9 and 0.8. A run with no number in its reply counts as a miss
for both.

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import sys; sys.path.append("./../..")
import re
from pathlib import Path
from statistics import median

import pandas as pd
from tqdm import tqdm
import ollama

from pipeline.utils.prompt_blocks import JOB_FIELDS, CV_FIELDS, SYSTEM_PROMPT, build_block

DATA_DIR   = Path("../../datasets/findhr")
JUDGE      = "ministral-3-14b"       # edit to switch judge
GEN_MODEL  = "ministral-3:14b"       # its Ollama model
JUDGE_DIR  = DATA_DIR / "judges" / JUDGE
K           = 5      # runs per pair; the label is the median of the K runs
TEMPERATURE = 0.6    # > 0 so the K runs differ; their disagreement is the confidence
CHUNK_SIZE  = 100    # pairs written per append
OUTPUT_FILE = JUDGE_DIR / "llm_relevance.csv"

print(f"judge {JUDGE}: {GEN_MODEL} -> {OUTPUT_FILE}")


In [ ]:
# Only the fields the judge is allowed to see.
jobs = pd.read_json(DATA_DIR / "source/filtered_jobs_dt.json")[
    ["id_j", "raw_occupation", "years_experience", "edu_eqf", "raw_skills", "eng_description", "languages"]
].set_index("id_j")

cvs = pd.read_json(DATA_DIR / "source/filtered_cvs_dt.json")[
    ["id_c", "years_experience", "edu_eqf", "raw_skills", "languages"]
].set_index("id_c")

# One row per (id_j, id_c) pair to judge.
pairs = pd.read_csv(DATA_DIR / "applications/applications_short.csv")

assert jobs.index.is_unique, "duplicate id_j in the jobs source"
assert cvs.index.is_unique, "duplicate id_c in the cvs source"
assert not pairs.duplicated().any(), "the same (id_j, id_c) pair appears twice in the pool"
assert pairs.id_j.isin(jobs.index).all(), "the pool references a job that is not in the source"
assert pairs.id_c.isin(cvs.index).all(), "the pool references a CV that is not in the source"

print(f"{len(jobs):,} jobs | {len(cvs):,} cvs | {len(pairs):,} pairs to judge "
      f"({pairs.id_j.nunique()} jobs x ~{len(pairs) / pairs.id_j.nunique():.0f} candidates)")

In [ ]:
job_blocks = {jid: build_block(jobs.loc[jid], JOB_FIELDS) for jid in pairs.id_j.unique()}
cv_blocks  = {cid: build_block(cvs.loc[cid], CV_FIELDS) for cid in pairs.id_c.unique()}

print(f"{len(job_blocks)} job blocks, {len(cv_blocks)} cv blocks, as the judge reads them:\n")
print(job_blocks[pairs.id_j.iloc[0]])
print("\n>>> CANDIDATE\n")
print(cv_blocks[pairs.id_c.iloc[0]])

In [ ]:
print(SYSTEM_PROMPT)

In [ ]:
def parse_reply(text: str) -> int | None:
    """'2' -> 2. Takes the first number in the reply and clamps it to 0..2.

    A reply with no number at all returns `None`: the run is discarded, not read as a 0.
    Scoring it 0 would turn "the model produced garbage" into a confident "not relevant",
    which the analysis cannot tell apart from a real judgement.
    """
    nums = re.findall(r"\d+", text)
    return min(max(int(nums[0]), 0), 2) if nums else None

def score_once(job_block: str, cv_block: str) -> int | None:
    """Relevance in {0, 1, 2} from one LLM call, or None if the reply held no number.

    think=False strips qwen3's
    <think> block; num_predict=8 is room for a single digit and nothing else.
    """
    user = (
        f"JOB OFFER:\n{job_block}\n\n"
        f"CANDIDATE CV:\n{cv_block}\n\n"
        "Relevance score:"
    )
    response = ollama.chat(
        model=GEN_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user},
        ],
        think=False,
        options={"temperature": TEMPERATURE, "num_predict": 8},
    )
    return parse_reply(response.message.content or "")

In [ ]:
def aggregate(runs) -> tuple[int | None, float, float]:
    """(label, agreement, stability) of the K runs of one pair.

    label:     median of the runs that produced a number; None if none did.
    agreement: share of runs landing exactly on the label: 1.0 / 0.8 / 0.6 with K=5.
                Coarse, but immediately readable ("4 runs out of 5 agreed").
    stability: 1 - mean|run - label| / 2. Same 0..1 direction, but aware that the scale
                is ordinal: agreement scores (2,2,2,2,1) and (2,2,2,2,0) both 0.8, while
                stability separates them (0.9 vs 0.8): dissenting by two grades is a
                wider disagreement than dissenting by one.

    A run whose reply held no number is not evidence for the label: it counts as a miss
    for agreement and as the maximum distance (2) for stability.
    """
    usable = [r for r in runs if r is not None]
    if not usable:
        return None, 0.0, 0.0
    label = int(median(usable))    # an even count (only if a run failed) rounds a .5 midpoint down
    deviations = [abs(r - label) if r is not None else 2 for r in runs]
    return (label,
            round(sum(r == label for r in runs) / len(runs), 3),
            round(1 - sum(deviations) / (2 * len(runs)), 3))


In [ ]:
RUN_COLS = [f"run_{i + 1}" for i in range(K)]
COLUMNS  = ["id_j", "id_c"] + RUN_COLS + ["median_relevance", "agreement", "stability"]

pair_key = pd.MultiIndex.from_frame(pairs[["id_j", "id_c"]])
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

if OUTPUT_FILE.exists():
    done = pd.read_csv(OUTPUT_FILE, usecols=["id_j", "id_c"])
    done_key = pd.MultiIndex.from_frame(done[["id_j", "id_c"]])
    todo = pairs[~pair_key.isin(done_key)]
    stale = (~done_key.isin(pair_key)).sum()
    print(f"resuming: {len(done):,} pairs already scored, {len(todo):,} left")
    if stale:
        print(f"  WARNING: {stale:,} scored rows belong to no pair in the current pool "
              f"(left over from an older applications_short.csv).\n"
              f"  They are ignored here but still sit in the file. Delete "
              f"{OUTPUT_FILE.name} and start over if the pool was regenerated.")
else:
    pd.DataFrame(columns=COLUMNS).to_csv(OUTPUT_FILE, index=False)
    todo = pairs
    print(f"starting fresh: {len(todo):,} pairs, ~{len(todo) * K:,} LLM calls")

In [ ]:
for start in range(0, len(todo), CHUNK_SIZE):
    chunk = todo.iloc[start: start + CHUNK_SIZE]

    rows = []
    for pair in tqdm(chunk.itertuples(index=False), total=len(chunk),
                     desc=f"pairs {start:,}-{start + len(chunk):,} of {len(todo):,}"):
        runs = [score_once(job_blocks[pair.id_j], cv_blocks[pair.id_c]) for _ in range(K)]
        label, agreement, stability = aggregate(runs)

        rows.append({"id_j": pair.id_j, "id_c": pair.id_c, **dict(zip(RUN_COLS, runs)),
                     "median_relevance": label, "agreement": agreement, "stability": stability})

    pd.DataFrame(rows, columns=COLUMNS).to_csv(OUTPUT_FILE, mode="a", index=False, header=False)